# Tema: AI Dungeon Master - Sang Juru Kisah

Final Project *LLM-Based Tools and Gemini API Integration for Data Scientists* (Hacktiv8).

Sang Juru Kisah adalah chatbot yang berperan sebagai Dungeon Master untuk permainan bergaya
Dungeons & Dragons 5e. Pemain bermain sendirian, DM-nya AI. Dibuat dengan Gemini, agent
LangGraph, dan Streamlit, lalu dijalankan di Colab dan dibuka lewat ngrok.

Ide utamanya sederhana: LLM yang bercerita, tapi semua angka (dadu, HP, AC, gold, XP) dihitung
oleh fungsi Python lewat tool calling. Gemini memilih tool yang mau dipanggil, kode yang menghitung hasilnya. Jadi DM tidak bisa
asal mengarang hasil dadu.

## Parameter kreatif

Semua bisa diatur dari sidebar app:

- **Setting dunia**: Nusantara Mistis (folklor Indonesia), High Fantasy Klasik, Horor Gothic, atau Bajak Laut & Lautan
- **Gaya narasi**: Epik & Dramatis, Santai & Kocak, atau Gelap & Mencekam
- **Tingkat kesulitan**: Mudah, Normal, atau Sulit. Ini mengatur besar DC dan kekuatan musuh
- **Bahasa**: Indonesia atau English
- **Panjang narasi** per giliran
- **Model Gemini dan temperature**

Fitur tambahannya ada dua. Pertama, integrasi API eksternal ke D&D 5e SRD API (dnd5eapi.co)
untuk data monster, spell, kondisi, dan item. Kedua, memory: DM ingat seluruh petualangan selama
sesi berjalan.

## Isi notebook

1. Setup: install library, ngrok, API key, dan helper `run_streamlit`
2. Part 1: mekanik permainan sebagai tools (`game_tools.py`)
3. Part 2: agent Dungeon Master (`dm_agent.py`)
4. Part 3: tampilan Streamlit (`app.py`)
5. Menjalankan app lewat ngrok

## Setup: Install Library & Konfigurasi ngrok

### Step 1 — Install Streamlit dan pyngrok

In [60]:
!pip install -q streamlit pyngrok langchain langchain-google-genai langgraph requests

### Step 2 — Daftarkan ngrok Auth Token

Cara mendapatkan token:
1. Daftar gratis di [https://ngrok.com](https://ngrok.com)
2. Login → klik **Your Authtoken** di dashboard
3. Copy token-nya, paste ke Colab Secrets dengan nama `NGROK_TOKEN`
   (klik ikon 🔑 di sidebar kiri Colab)

> Token ngrok gratis sudah cukup untuk keperluan training.
> Satu akun ngrok bisa membuka 1 tunnel aktif sekaligus.

In [61]:
! rm -f /root/.config/ngrok/ngrok.yml

import os
from pyngrok import ngrok
from google.colab import userdata

# Ambil token dari Colab Secrets
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
print("ngrok token berhasil dikonfigurasi!")

# API key Gemini (bisa diketik lewat sidebar app)
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI')
    print("API key Gemini berhasil diambil dari Colab Secrets!")
except Exception:
    print("Secret GEMINI tidak ditemukan. Masukkan API key di sidebar app nanti.")

ngrok token berhasil dikonfigurasi!
API key Gemini berhasil diambil dari Colab Secrets!


### Fungsi Helper: Jalankan Streamlit + Buka Tunnel

Kita buat fungsi reusable yang akan kita pakai berulang kali di setiap bagian.
Fungsi ini melakukan tiga hal:
1. Menulis file `.py` ke disk
2. Menjalankan `streamlit run` di background
3. Membuka tunnel ngrok dan menampilkan URL publik

In [62]:
import subprocess
import time

def run_streamlit(filename, port=8501):
    # Kill SEMUA proses streamlit, bukan hanya yang kita spawn
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

    # Force-free port kalau masih ada yang nempel
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)

    # Tutup semua tunnel ngrok
    ngrok.kill()

    # Tunggu port benar-benar bebas
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=os.environ.copy(),   # teruskan GOOGLE_API_KEY ke app
    )

    # LangChain butuh waktu sedikit lebih lama untuk di-load
    time.sleep(6)

    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")

    return proc

---
# Part 1: Game Tools (`game_tools.py`)

LLM bagus untuk bercerita, tetapi angka bisa salah. Kalau dibiarkan, ia bisa lupa
HP karakter, membuat lemparan dadu yang selalu menguntungkan pemain, atau menambah gold entah
dari mana. Solusinya sama dengan SQL Agent: LLM tidak menghitung sendiri, tapi memanggil
fungsi Python dan memakai hasilnya.

Ada 19 tools di file ini:

| Kelompok | Tools |
|---|---|
| Dadu | `roll_dice`, `ability_check` |
| Karakter | `create_character`, `get_game_state`, `update_hp`, `update_inventory`, `update_gold`, `set_condition`, `take_rest`, `award_xp` |
| Dunia | `set_location`, `update_quest` |
| Combat | `add_enemy`, `roll_initiative`, `player_attack`, `damage_enemy`, `enemy_attack`, `end_combat` |
| API eksternal | `lookup_srd`, plus `add_enemy` kalau diisi `srd_monster` |

Tiap pemain punya objek `GameState` sendiri. Fungsi `build_tools(state)` membuat tools
yang terikat ke state pemain itu (closure), dan `GameState` disimpan di `st.session_state`.

Decorator `@tool` membaca docstring dan type hint, termasuk `Literal[...]`, lalu mengubahnya
jadi skema yang dibaca Gemini. Jadi docstring di sini juga berfungsi sebagai petunjuk pemakaian
untuk LLM.

In [63]:
%%writefile game_tools.py
"""
game_tools.py
=============
Semua MEKANIK permainan ada di sini: lempar dadu, karakter, HP, inventaris,
quest, combat, XP, dan pencarian aturan dari D&D 5e SRD API.

Prinsip utamanya: LLM hanya bercerita, angka dihitung oleh KODE.
LLM (Gemini) tidak boleh mengarang hasil dadu atau HP; ia wajib memanggil
tools di file ini.

Setiap sesi Streamlit punya satu objek `GameState`. Tools dibuat sebagai
closure yang "mengikat" objek tersebut lewat `build_tools(state)`, sehingga
dua pemain di dua tab browser tidak saling mengganggu.
"""

from __future__ import annotations

import random
import re
from dataclasses import dataclass, field
from typing import Literal, Optional

import requests
from langchain_core.tools import tool

# ─────────────────────────────────────────────────────────────────────────────
# 1. Data aturan (diambil dari SRD 5.1, lisensi CC-BY-4.0)
# ─────────────────────────────────────────────────────────────────────────────

ABILITIES = ["STR", "DEX", "CON", "INT", "WIS", "CHA"]
STANDARD_ARRAY = [15, 14, 13, 12, 10, 8]

# Ambang XP untuk naik level (SRD 5.1)
XP_THRESHOLDS = [0, 300, 900, 2700, 6500, 14000, 23000, 34000, 48000, 64000,
                 85000, 100000, 120000, 140000, 165000, 195000, 225000,
                 265000, 305000, 355000]

ClassName = Literal["Barbarian", "Bard", "Cleric", "Druid", "Fighter", "Monk",
                    "Paladin", "Ranger", "Rogue", "Sorcerer", "Warlock", "Wizard"]

# armor: (nama, jenis, nilai dasar)
#   jenis "heavy" -> AC = dasar
#   jenis "medium" -> AC = dasar + min(DEX, 2)
#   jenis "light" -> AC = dasar + DEX
#   jenis "none" -> AC = 10 + DEX
#   jenis "barbarian"/"monk" -> unarmored defense
CLASS_DATA: dict[str, dict] = {
    "Barbarian": dict(hit_die=12, priority=["STR", "CON", "DEX", "WIS", "CHA", "INT"],
                      armor=("Tanpa armor", "barbarian", 10), shield=False, spell=None,
                      weapon=("Greataxe", "1d12", "STR"),
                      items={"Greataxe": 1, "Handaxe": 2, "Javelin": 4, "Explorer's Pack": 1}),
    "Bard": dict(hit_die=8, priority=["CHA", "DEX", "CON", "WIS", "INT", "STR"],
                 armor=("Leather Armor", "light", 11), shield=False, spell="CHA",
                 weapon=("Rapier", "1d8", "DEX"),
                 items={"Rapier": 1, "Dagger": 1, "Lute": 1, "Leather Armor": 1, "Entertainer's Pack": 1}),
    "Cleric": dict(hit_die=8, priority=["WIS", "CON", "STR", "DEX", "CHA", "INT"],
                   armor=("Scale Mail", "medium", 14), shield=True, spell="WIS",
                   weapon=("Mace", "1d6", "STR"),
                   items={"Mace": 1, "Scale Mail": 1, "Shield": 1, "Holy Symbol": 1, "Priest's Pack": 1}),
    "Druid": dict(hit_die=8, priority=["WIS", "CON", "DEX", "INT", "CHA", "STR"],
                  armor=("Leather Armor", "light", 11), shield=True, spell="WIS",
                  weapon=("Scimitar", "1d6", "DEX"),
                  items={"Scimitar": 1, "Wooden Shield": 1, "Leather Armor": 1, "Druidic Focus": 1, "Explorer's Pack": 1}),
    "Fighter": dict(hit_die=10, priority=["STR", "CON", "DEX", "WIS", "CHA", "INT"],
                    armor=("Chain Mail", "heavy", 16), shield=True, spell=None,
                    weapon=("Longsword", "1d8", "STR"),
                    items={"Longsword": 1, "Shield": 1, "Chain Mail": 1, "Light Crossbow": 1, "Dungeoneer's Pack": 1}),
    "Monk": dict(hit_die=8, priority=["DEX", "WIS", "CON", "STR", "INT", "CHA"],
                 armor=("Tanpa armor", "monk", 10), shield=False, spell=None,
                 weapon=("Shortsword", "1d6", "DEX"),
                 items={"Shortsword": 1, "Dart": 10, "Explorer's Pack": 1}),
    "Paladin": dict(hit_die=10, priority=["STR", "CHA", "CON", "WIS", "DEX", "INT"],
                    armor=("Chain Mail", "heavy", 16), shield=True, spell="CHA",
                    weapon=("Longsword", "1d8", "STR"),
                    items={"Longsword": 1, "Shield": 1, "Chain Mail": 1, "Holy Symbol": 1, "Priest's Pack": 1}),
    "Ranger": dict(hit_die=10, priority=["DEX", "WIS", "CON", "STR", "INT", "CHA"],
                   armor=("Scale Mail", "medium", 14), shield=False, spell="WIS",
                   weapon=("Longbow", "1d8", "DEX"),
                   items={"Longbow": 1, "Arrow": 20, "Shortsword": 2, "Scale Mail": 1, "Explorer's Pack": 1}),
    "Rogue": dict(hit_die=8, priority=["DEX", "CON", "INT", "WIS", "CHA", "STR"],
                  armor=("Leather Armor", "light", 11), shield=False, spell=None,
                  weapon=("Shortsword", "1d6", "DEX"),
                  items={"Shortsword": 1, "Shortbow": 1, "Arrow": 20, "Dagger": 2, "Thieves' Tools": 1, "Leather Armor": 1}),
    "Sorcerer": dict(hit_die=6, priority=["CHA", "CON", "DEX", "WIS", "INT", "STR"],
                     armor=("Tanpa armor", "none", 10), shield=False, spell="CHA",
                     weapon=("Dagger", "1d4", "DEX"),
                     items={"Dagger": 2, "Arcane Focus": 1, "Light Crossbow": 1, "Dungeoneer's Pack": 1}),
    "Warlock": dict(hit_die=8, priority=["CHA", "CON", "DEX", "WIS", "INT", "STR"],
                    armor=("Leather Armor", "light", 11), shield=False, spell="CHA",
                    weapon=("Dagger", "1d4", "DEX"),
                    items={"Dagger": 2, "Arcane Focus": 1, "Leather Armor": 1, "Scholar's Pack": 1}),
    "Wizard": dict(hit_die=6, priority=["INT", "CON", "DEX", "WIS", "CHA", "STR"],
                   armor=("Tanpa armor", "none", 10), shield=False, spell="INT",
                   weapon=("Quarterstaff", "1d6", "STR"),
                   items={"Quarterstaff": 1, "Spellbook": 1, "Arcane Focus": 1, "Scholar's Pack": 1}),
}

SRD_API_BASES = ["https://www.dnd5eapi.co/api/2014", "https://www.dnd5eapi.co/api"]


def ability_mod(score: int) -> int:
    """Modifier ability score D&D: (skor - 10) // 2."""
    return (score - 10) // 2


def fmt_mod(value: int) -> str:
    return f"+{value}" if value >= 0 else str(value)


# ─────────────────────────────────────────────────────────────────────────────
# 2. Mesin dadu
# ─────────────────────────────────────────────────────────────────────────────

_TERM_RE = re.compile(r"([+-])?(?:(\d*)d(\d+)(?:(kh|kl)(\d+))?|(\d+))", re.IGNORECASE)


def roll_expression(expr: str, rng: random.Random, crit: bool = False) -> tuple[int, str]:
    """Lempar ekspresi dadu seperti '1d20+5', '2d6+1d4-1', '4d6kh3', '2d20kl1'.

    crit=True menggandakan JUMLAH dadu (aturan critical hit), bukan modifier.
    Mengembalikan (total, rincian_teks).
    """
    clean = expr.replace(" ", "").lower()
    if not clean:
        raise ValueError("Notasi dadu kosong.")

    pos, total, parts = 0, 0, []
    for m in _TERM_RE.finditer(clean):
        if m.start() != pos:
            raise ValueError(f"Notasi dadu tidak valid: '{expr}'")
        pos = m.end()
        sign = -1 if m.group(1) == "-" else 1
        sign_txt = "-" if sign < 0 else "+"

        if m.group(3):  # bagian dadu, mis. 2d6
            count = int(m.group(2) or 1)
            sides = int(m.group(3))
            if crit:
                count *= 2
            if not (1 <= count <= 100 and 2 <= sides <= 1000):
                raise ValueError("Jumlah dadu 1-100 dan sisi dadu 2-1000.")
            rolls = [rng.randint(1, sides) for _ in range(count)]
            kept = rolls
            if m.group(4):  # keep highest / keep lowest
                k = max(1, min(int(m.group(5)), count))
                kept = sorted(rolls, reverse=(m.group(4) == "kh"))[:k]
            subtotal = sum(kept)
            total += sign * subtotal
            note = f"{count}d{sides}{(m.group(4) or '') + (m.group(5) or '')}{rolls}"
            if kept is not rolls:
                note += f"→ambil {kept}"
            parts.append(f"{sign_txt} {note}")
        else:  # konstanta, mis. +5
            value = int(m.group(6))
            total += sign * value
            parts.append(f"{sign_txt} {value}")

    if pos != len(clean):
        raise ValueError(f"Notasi dadu tidak valid: '{expr}'")

    detail = " ".join(parts).lstrip("+ ").strip()
    return total, detail


def roll_d20(rng: random.Random, advantage: str = "normal") -> tuple[int, str]:
    """Lempar d20 dengan advantage/disadvantage. Mengembalikan (hasil, rincian)."""
    if advantage == "advantage":
        a, b = rng.randint(1, 20), rng.randint(1, 20)
        return max(a, b), f"d20 advantage [{a}, {b}] → {max(a, b)}"
    if advantage == "disadvantage":
        a, b = rng.randint(1, 20), rng.randint(1, 20)
        return min(a, b), f"d20 disadvantage [{a}, {b}] → {min(a, b)}"
    r = rng.randint(1, 20)
    return r, f"d20 [{r}]"


# ─────────────────────────────────────────────────────────────────────────────
# 3. State permainan
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class Character:
    name: str
    species: str
    char_class: str
    background: str
    abilities: dict[str, int]
    max_hp: int
    hp: int
    ac: int
    armor_name: str
    weapon: tuple[str, str, str]          # (nama, dadu damage, ability)
    spell_ability: Optional[str]
    level: int = 1
    xp: int = 0
    gold: int = 15
    hit_dice_left: int = 1
    inventory: dict[str, int] = field(default_factory=dict)
    conditions: list[str] = field(default_factory=list)

    @property
    def prof_bonus(self) -> int:
        return 2 + (self.level - 1) // 4

    def mod(self, ability: str) -> int:
        return ability_mod(self.abilities[ability.upper()])

    @property
    def hit_die(self) -> int:
        return CLASS_DATA[self.char_class]["hit_die"]


@dataclass
class Enemy:
    name: str
    max_hp: int
    hp: int
    ac: int
    attack_bonus: int
    damage_dice: str
    dex_mod: int = 0
    xp: int = 0


@dataclass
class GameState:
    character: Optional[Character] = None
    enemies: dict[str, Enemy] = field(default_factory=dict)
    initiative_order: list[tuple[str, int]] = field(default_factory=list)
    quests: dict[str, dict] = field(default_factory=dict)
    location: str = "Belum ditentukan"
    roll_log: list[str] = field(default_factory=list)
    rng: random.Random = field(default_factory=random.Random)

    def log(self, text: str) -> str:
        """Catat kejadian mekanik (ditampilkan di sidebar UI)."""
        self.roll_log.append(text)
        self.roll_log[:] = self.roll_log[-30:]
        return text

    def find_enemy(self, name: str) -> Optional[Enemy]:
        if name in self.enemies:
            return self.enemies[name]
        lowered = name.lower().strip()
        for key, enemy in self.enemies.items():
            if key.lower() == lowered:
                return enemy
        matches = [e for k, e in self.enemies.items() if lowered in k.lower()]
        return matches[0] if len(matches) == 1 else None

    def summary(self) -> str:
        """Ringkasan state lengkap untuk LLM."""
        lines = [f"Lokasi: {self.location}"]
        c = self.character
        if c is None:
            lines.append("Karakter: BELUM DIBUAT. Pandu pemain membuat karakter dulu.")
        else:
            abil = ", ".join(f"{a} {c.abilities[a]} ({fmt_mod(c.mod(a))})" for a in ABILITIES)
            inv = ", ".join(f"{k} x{v}" for k, v in c.inventory.items()) or "kosong"
            next_xp = XP_THRESHOLDS[c.level] if c.level < 20 else "-"
            spell = ""
            if c.spell_ability:
                spell = (f"\nSpellcasting: {c.spell_ability}, spell attack "
                         f"{fmt_mod(c.mod(c.spell_ability) + c.prof_bonus)}, "
                         f"spell save DC {8 + c.prof_bonus + c.mod(c.spell_ability)}")
            lines.append(
                f"Karakter: {c.name}, {c.species} {c.char_class} level {c.level} "
                f"(XP {c.xp}/{next_xp})\n"
                f"Background: {c.background or '-'}\n"
                f"HP {c.hp}/{c.max_hp} | AC {c.ac} ({c.armor_name}) | "
                f"Proficiency {fmt_mod(c.prof_bonus)} | Gold {c.gold} gp | "
                f"Hit dice tersisa {c.hit_dice_left}d{c.hit_die}\n"
                f"Ability: {abil}\n"
                f"Senjata utama: {c.weapon[0]} ({c.weapon[1]} + {c.weapon[2]})"
                f"{spell}\n"
                f"Inventaris: {inv}\n"
                f"Kondisi: {', '.join(c.conditions) or 'normal'}"
            )
        if self.enemies:
            foes = "; ".join(f"{e.name} HP {e.hp}/{e.max_hp} AC {e.ac}"
                             for e in self.enemies.values())
            lines.append(f"Musuh aktif: {foes}")
            if self.initiative_order:
                order = " → ".join(f" {n} ({v})" for n, v in self.initiative_order)
                lines.append(f"Urutan inisiatif: {order}")
        else:
            lines.append("Musuh aktif: tidak ada (bukan mode combat)")
        if self.quests:
            q = "; ".join(f"[{v['status']}] {k}: {v['note']}" for k, v in self.quests.items())
            lines.append(f"Quest: {q}")
        return "\n".join(lines)


# ─────────────────────────────────────────────────────────────────────────────
# 4. Integrasi API eksternal: D&D 5e SRD API (dnd5eapi.co)
# ─────────────────────────────────────────────────────────────────────────────

def slugify(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")


def srd_get(path: str, timeout: float = 8.0) -> Optional[dict]:
    """GET ke dnd5eapi.co, mencoba endpoint /api/2014 lalu /api (fallback)."""
    for base in SRD_API_BASES:
        try:
            resp = requests.get(f"{base}/{path.lstrip('/')}", timeout=timeout)
            if resp.status_code == 200:
                return resp.json()
        except requests.RequestException:
            continue
    return None


def srd_find(category: str, name: str) -> Optional[dict]:
    """Cari entri SRD berdasarkan nama: coba index langsung, lalu pencarian nama."""
    data = srd_get(f"{category}/{slugify(name)}")
    if data and "index" in data:
        return data
    listing = srd_get(f"{category}?name={requests.utils.quote(name)}")
    if listing and listing.get("results"):
        return srd_get(f"{category}/{listing['results'][0]['index']}")
    return None


def _desc(data: dict, limit: int = 900) -> str:
    desc = data.get("desc", "")
    if isinstance(desc, list):
        desc = " ".join(desc)
    return desc[:limit] + ("…" if len(desc) > limit else "")


def format_srd(category: str, data: dict) -> str:
    """Ringkas JSON SRD menjadi teks pendek yang mudah dipakai LLM."""
    name = data.get("name", "?")
    if category == "spells":
        dmg = data.get("damage", {})
        dmg_txt = ""
        if isinstance(dmg, dict):
            if dmg.get("damage_at_slot_level"):
                dmg_txt = f"Damage per slot: {dmg['damage_at_slot_level']}"
            elif dmg.get("damage_at_character_level"):
                dmg_txt = f"Damage per level karakter: {dmg['damage_at_character_level']}"
        elif isinstance(dmg, list) and dmg:
            dmg_txt = f"Damage: {dmg}"
        save = data.get("dc", {}) or {}
        save_txt = f"Saving throw: {save.get('dc_type', {}).get('name', '')}" if save else ""
        return (f"{name} (level {data.get('level')}, {data.get('school', {}).get('name', '')})\n"
                f"Casting time {data.get('casting_time')}, range {data.get('range')}, "
                f"durasi {data.get('duration')}, komponen {''.join(data.get('components', []))}\n"
                f"{dmg_txt} {save_txt}\n{_desc(data)}")
    if category == "monsters":
        ac = data.get("armor_class", [{}])
        ac_val = ac[0].get("value") if isinstance(ac, list) and ac else ac
        actions = []
        for a in data.get("actions", [])[:4]:
            dmg = ", ".join(d.get("damage_dice", "") for d in a.get("damage", []) if "damage_dice" in d)
            bonus = a.get("attack_bonus")
            actions.append(f"{a.get('name')}" + (f" (+{bonus} to hit, {dmg})" if bonus is not None else ""))
        stats = " ".join(f"{k[:3].upper()} {data.get(k)}" for k in
                         ["strength", "dexterity", "constitution", "intelligence", "wisdom", "charisma"])
        hp_roll = data.get("hit_points_roll") or data.get("hit_dice", "")
        return (f"{name}: {data.get('size')} {data.get('type')}, CR {data.get('challenge_rating')}, "
                f"XP {data.get('xp')}\nAC {ac_val}, HP {data.get('hit_points')} ({hp_roll})\n"
                f"{stats}\nAksi: {'; '.join(actions) or '-'}")
    if category == "equipment":
        dmg = data.get("damage", {}) or {}
        extra = f"Damage {dmg.get('damage_dice')} {dmg.get('damage_type', {}).get('name', '')}" if dmg else ""
        ac = data.get("armor_class")
        if ac:
            extra += f" AC dasar {ac.get('base')}" + (" + DEX" if ac.get("dex_bonus") else "")
        cost = data.get("cost", {})
        return f"{name}: {data.get('equipment_category', {}).get('name', '')}. {extra}. Harga {cost.get('quantity', '?')} {cost.get('unit', '')}\n{_desc(data)}"
    return f"{name}\n{_desc(data, 1200)}"


# ─────────────────────────────────────────────────────────────────────────────
# 5. Tools untuk agent
# ─────────────────────────────────────────────────────────────────────────────

def build_tools(state: GameState) -> list:
    """Buat semua tools yang terikat ke satu GameState (satu sesi pemain)."""

    def need_character() -> Optional[str]:
        if state.character is None:
            return "ERROR: Karakter belum dibuat. Panggil create_character terlebih dahulu."
        return None

    # ── Dadu & cek kemampuan ────────────────────────────────────────────────
    @tool
    def roll_dice(notation: str, reason: str) -> str:
        """Lempar dadu dengan notasi standar, mis. '1d20+3', '2d6+1d4', '4d6kh3'.
        Wajib dipakai untuk SETIAP lemparan dadu yang tidak tercakup tool lain.

        Args:
            notation: notasi dadu.
            reason: untuk apa lemparan ini (mis. 'damage fireball', 'tebakan acak NPC').
        """
        try:
            total, detail = roll_expression(notation, state.rng)
        except ValueError as e:
            return f"ERROR: {e}"
        return state.log(f"🎲 {reason}: {notation} → {detail} = **{total}**")

    @tool
    def ability_check(ability: Literal["STR", "DEX", "CON", "INT", "WIS", "CHA"],
                      dc: int,
                      skill_or_purpose: str,
                      proficient: bool = False,
                      advantage: Literal["normal", "advantage", "disadvantage"] = "normal",
                      kind: Literal["check", "save"] = "check") -> str:
        """Lakukan ability check atau saving throw karakter pemain melawan DC.
        Modifier dihitung otomatis dari character sheet.

        Args:
            ability: ability yang dipakai.
            dc: Difficulty Class (5 sangat mudah, 10 mudah, 15 sedang, 20 sulit, 25 sangat sulit).
            skill_or_purpose: nama skill atau tujuan, mis. 'Stealth', 'Perception', 'menahan racun'.
            proficient: True jika karakter mahir di skill/saving throw ini.
            advantage: 'advantage', 'disadvantage', atau 'normal'.
            kind: 'check' untuk ability/skill check, 'save' untuk saving throw.
        """
        if err := need_character():
            return err
        c = state.character
        roll, detail = roll_d20(state.rng, advantage)
        bonus = c.mod(ability) + (c.prof_bonus if proficient else 0)
        total = roll + bonus
        success = total >= dc
        label = "Saving throw" if kind == "save" else "Check"
        verdict = "BERHASIL ✅" if success else "GAGAL ❌"
        if roll == 20:
            verdict += " (natural 20!)"
        elif roll == 1:
            verdict += " (natural 1!)"
        return state.log(f"🎲 {label} {ability} ({skill_or_purpose}): {detail} {fmt_mod(bonus)} "
                         f"= **{total}** vs DC {dc} → {verdict}")

    # ── Karakter ─────────────────────────────────────────────────────────────
    @tool
    def create_character(name: str,
                         species: str,
                         char_class: ClassName,
                         background: str = "",
                         stat_method: Literal["standard", "roll"] = "standard") -> str:
        """Buat karakter pemain level 1. Ability score, HP, AC, senjata, dan
        perlengkapan awal dihitung otomatis sesuai class.

        Args:
            name: nama karakter.
            species: ras/spesies, mis. Human, Elf, Dwarf, Halfling, Gnome, Half-Orc, Tiefling, Dragonborn.
            char_class: class D&D (nama bahasa Inggris).
            background: latar belakang singkat karakter (opsional).
            stat_method: 'standard' (15,14,13,12,10,8) atau 'roll' (4d6 buang terendah).
        """
        data = CLASS_DATA[char_class]
        roll_note = ""
        if stat_method == "roll":
            scores = []
            for _ in range(6):
                rolls = [state.rng.randint(1, 6) for _ in range(4)]
                scores.append(sum(sorted(rolls)[1:]))
            scores.sort(reverse=True)
            roll_note = f" (hasil 4d6 drop lowest: {scores})"
        else:
            scores = list(STANDARD_ARRAY)
        abilities = {ab: scores[i] for i, ab in enumerate(data["priority"])}
        abilities = {ab: abilities[ab] for ab in ABILITIES}

        dex, con, wis = (ability_mod(abilities[a]) for a in ("DEX", "CON", "WIS"))
        armor_name, armor_kind, base = data["armor"]
        ac = {"heavy": base, "medium": base + min(dex, 2), "light": base + dex,
              "none": 10 + dex, "barbarian": 10 + dex + con, "monk": 10 + dex + wis}[armor_kind]
        if data["shield"]:
            ac += 2
            armor_name += " + Shield"
        max_hp = max(1, data["hit_die"] + con)

        state.character = Character(
            name=name, species=species, char_class=char_class, background=background,
            abilities=abilities, max_hp=max_hp, hp=max_hp, ac=ac, armor_name=armor_name,
            weapon=data["weapon"], spell_ability=data["spell"],
            inventory=dict(data["items"]), hit_dice_left=1,
        )
        state.log(f"🧙 Karakter dibuat: {name}, {species} {char_class}{roll_note}")
        return "Karakter berhasil dibuat.\n" + state.summary()

    @tool
    def get_game_state() -> str:
        """Lihat state permainan saat ini: character sheet, inventaris, kondisi,
        musuh aktif, urutan inisiatif, quest, dan lokasi. Panggil di awal sesi
        atau kapan pun kamu ragu tentang angka/fakta permainan."""
        return state.summary()

    @tool
    def update_hp(amount: int, reason: str) -> str:
        """Ubah HP karakter pemain. Angka negatif = damage, positif = heal.
        Gunakan untuk damage dari jebakan/lingkungan/efek, atau heal dari potion/spell.
        (Damage dari serangan musuh sudah otomatis lewat enemy_attack.)

        Args:
            amount: perubahan HP (mis. -7 atau +5).
            reason: penyebabnya.
        """
        if err := need_character():
            return err
        c = state.character
        before = c.hp
        c.hp = max(0, min(c.max_hp, c.hp + amount))
        msg = f"❤️ HP {c.name}: {before} → {c.hp}/{c.max_hp} ({fmt_mod(amount)}, {reason})"
        if c.hp == 0 and "Unconscious" not in c.conditions:
            c.conditions.append("Unconscious")
            msg += ". TUMBANG (Unconscious)! Mulai death saving throw."
        elif c.hp > 0 and "Unconscious" in c.conditions:
            c.conditions.remove("Unconscious")
            msg += ". Sadar kembali."
        return state.log(msg)

    @tool
    def update_inventory(item: str, quantity_change: int) -> str:
        """Tambah (angka positif) atau kurangi (angka negatif) item di inventaris.

        Args:
            item: nama item.
            quantity_change: mis. 1 untuk menambah satu, -1 untuk memakai/menjual satu.
        """
        if err := need_character():
            return err
        inv = state.character.inventory
        key = next((k for k in inv if k.lower() == item.lower()), item)
        current = inv.get(key, 0)
        if current + quantity_change < 0:
            return f"ERROR: {key} hanya ada {current}, tidak bisa dikurangi {abs(quantity_change)}."
        new = current + quantity_change
        if new == 0:
            inv.pop(key, None)
        else:
            inv[key] = new
        return state.log(f"🎒 {key}: {current} → {new}")

    @tool
    def update_gold(amount: int, reason: str) -> str:
        """Tambah/kurangi gold (gp) karakter. Negatif untuk membayar.

        Args:
            amount: perubahan gold.
            reason: alasannya (mis. 'hadiah quest', 'beli potion').
        """
        if err := need_character():
            return err
        c = state.character
        if c.gold + amount < 0:
            return f"ERROR: Gold tidak cukup. Saat ini {c.gold} gp, butuh {abs(amount)} gp."
        before = c.gold
        c.gold += amount
        return state.log(f"💰 Gold: {before} → {c.gold} gp ({reason})")

    @tool
    def set_condition(condition: str, active: bool) -> str:
        """Pasang atau hapus kondisi pada karakter (mis. Poisoned, Frightened, Prone, Blinded).

        Args:
            condition: nama kondisi (gunakan nama SRD dalam bahasa Inggris).
            active: True untuk memasang, False untuk menghapus.
        """
        if err := need_character():
            return err
        conds = state.character.conditions
        cond = condition.strip().title()
        if active and cond not in conds:
            conds.append(cond)
        elif not active and cond in conds:
            conds.remove(cond)
        return state.log(f"⚠️ Kondisi {cond}: {'aktif' if active else 'hilang'}")

    @tool
    def take_rest(kind: Literal["short", "long"]) -> str:
        """Karakter beristirahat. Short rest: pakai 1 hit die untuk heal.
        Long rest: HP penuh, hit dice pulih, kondisi ringan hilang."""
        if err := need_character():
            return err
        c = state.character
        if kind == "long":
            c.hp = c.max_hp
            c.hit_dice_left = c.level
            c.conditions = [x for x in c.conditions if x not in
                            ("Poisoned", "Frightened", "Unconscious", "Prone", "Exhaustion")]
            return state.log(f"🏕️ Long rest: HP penuh {c.hp}/{c.max_hp}, hit dice pulih.")
        if c.hp >= c.max_hp:
            return state.log(f"☕ Short rest: HP sudah penuh ({c.hp}/{c.max_hp}), hit dice tidak dipakai.")
        if c.hit_dice_left <= 0:
            return "Tidak ada hit dice tersisa. Butuh long rest."
        heal = max(1, state.rng.randint(1, c.hit_die) + c.mod("CON"))
        c.hit_dice_left -= 1
        before = c.hp
        c.hp = min(c.max_hp, c.hp + heal)
        return state.log(f"☕ Short rest: heal {heal}, HP {before} → {c.hp}/{c.max_hp}")

    @tool
    def award_xp(amount: int, reason: str) -> str:
        """Beri XP ke karakter (setelah mengalahkan musuh, menyelesaikan quest,
        atau roleplay yang bagus). Naik level dihitung otomatis.

        Args:
            amount: jumlah XP.
            reason: alasannya.
        """
        if err := need_character():
            return err
        c = state.character
        c.xp += amount
        msg = f"⭐ +{amount} XP ({reason}). Total {c.xp} XP."
        while c.level < 20 and c.xp >= XP_THRESHOLDS[c.level]:
            c.level += 1
            gain = max(1, c.hit_die // 2 + 1 + c.mod("CON"))
            c.max_hp += gain
            c.hp += gain
            c.hit_dice_left += 1
            msg += f" 🎉 NAIK LEVEL {c.level}! Max HP +{gain} → {c.max_hp}."
        return state.log(msg)

    # ── Dunia & quest ────────────────────────────────────────────────────────
    @tool
    def set_location(name: str) -> str:
        """Catat lokasi karakter saat ini setiap kali pindah tempat."""
        state.location = name
        return state.log(f"📍 Lokasi: {name}")

    @tool
    def update_quest(title: str, status: Literal["aktif", "selesai", "gagal"], note: str = "") -> str:
        """Tambah atau perbarui quest di quest log.

        Args:
            title: judul quest singkat.
            status: 'aktif', 'selesai', atau 'gagal'.
            note: tujuan/progres singkat.
        """
        prev = state.quests.get(title, {})
        state.quests[title] = {"status": status, "note": note or prev.get("note", "")}
        return state.log(f"📜 Quest '{title}': {status}")

    # ── Combat ───────────────────────────────────────────────────────────────
    @tool
    def add_enemy(name: str,
                  srd_monster: Optional[str] = None,
                  hp: Optional[int] = None,
                  ac: Optional[int] = None,
                  attack_bonus: Optional[int] = None,
                  damage_dice: Optional[str] = None) -> str:
        """Masukkan musuh ke arena combat.
        Isi `srd_monster` (nama monster SRD bahasa Inggris, mis. 'goblin', 'wolf',
        'skeleton', 'bandit') agar statistik diambil otomatis dari D&D 5e SRD API.
        Untuk musuh orisinal, isi hp, ac, attack_bonus, dan damage_dice sendiri.

        Args:
            name: nama unik musuh di cerita, mis. 'Goblin 1' atau 'Kuntilanak Rawa'.
            srd_monster: nama monster SRD (opsional).
            hp: hit points (wajib jika tanpa srd_monster).
            ac: armor class.
            attack_bonus: bonus serangan utama.
            damage_dice: damage serangan utama, mis. '1d6+2'.
        """
        dex_mod, xp, source = 0, 0, "custom"
        if srd_monster:
            data = srd_find("monsters", srd_monster)
            if data:
                source = "SRD API"
                hp = hp or data.get("hit_points")
                acs = data.get("armor_class", [{}])
                ac = ac or (acs[0].get("value") if isinstance(acs, list) and acs else acs)
                dex_mod = ability_mod(data.get("dexterity", 10))
                xp = data.get("xp", 0)
                for action in data.get("actions", []):
                    if action.get("attack_bonus") is not None:
                        attack_bonus = attack_bonus if attack_bonus is not None else action["attack_bonus"]
                        dice = [d.get("damage_dice") for d in action.get("damage", []) if d.get("damage_dice")]
                        damage_dice = damage_dice or (dice[0] if dice else None)
                        break
        if hp is None or ac is None:
            return ("ERROR: Statistik musuh tidak lengkap (SRD API tidak tersedia atau monster tidak "
                    "ditemukan). Panggil ulang dengan hp, ac, attack_bonus, dan damage_dice.")
        enemy = Enemy(name=name, max_hp=int(hp), hp=int(hp), ac=int(ac),
                      attack_bonus=int(attack_bonus if attack_bonus is not None else 3),
                      damage_dice=damage_dice or "1d6+1", dex_mod=dex_mod, xp=int(xp or 0))
        state.enemies[name] = enemy
        return state.log(f"👹 Musuh muncul: {name} (HP {enemy.hp}, AC {enemy.ac}, "
                         f"serangan {fmt_mod(enemy.attack_bonus)} / {enemy.damage_dice}) [{source}]")

    @tool
    def roll_initiative() -> str:
        """Lempar inisiatif untuk pemain dan semua musuh aktif, lalu tetapkan urutan giliran.
        Panggil sekali di awal setiap combat setelah semua musuh ditambahkan."""
        if err := need_character():
            return err
        order = []
        c = state.character
        r = state.rng.randint(1, 20)
        order.append((c.name, r + c.mod("DEX")))
        for e in state.enemies.values():
            order.append((e.name, state.rng.randint(1, 20) + e.dex_mod))
        order.sort(key=lambda x: x[1], reverse=True)
        state.initiative_order = order
        return state.log("⚔️ Inisiatif: " + " → ".join(f"{n} ({v})" for n, v in order))

    @tool
    def player_attack(target: str,
                      attack_type: Literal["weapon", "spell"] = "weapon",
                      damage_dice: Optional[str] = None,
                      advantage: Literal["normal", "advantage", "disadvantage"] = "normal",
                      description: str = "") -> str:
        """Pemain menyerang musuh. Attack roll vs AC dan damage dihitung otomatis;
        HP musuh langsung dikurangi. Natural 20 = critical hit (dadu damage ganda).

        Args:
            target: nama musuh (sesuai add_enemy).
            attack_type: 'weapon' memakai senjata utama, 'spell' memakai spell attack.
            damage_dice: override dadu damage (wajib untuk spell, mis. '1d10' Fire Bolt;
                         untuk weapon kosongkan agar memakai senjata utama).
            advantage: 'advantage', 'disadvantage', atau 'normal'.
            description: deskripsi singkat serangannya.
        """
        if err := need_character():
            return err
        c = state.character
        enemy = state.find_enemy(target)
        if enemy is None:
            names = ", ".join(state.enemies) or "tidak ada"
            return f"ERROR: Musuh '{target}' tidak ditemukan. Musuh aktif: {names}."

        if attack_type == "spell":
            if not c.spell_ability:
                return f"ERROR: {c.char_class} tidak punya spellcasting di level ini."
            ability = c.spell_ability
            dmg_expr = damage_dice or "1d8"
            dmg_bonus = 0
        else:
            ability = c.weapon[2]
            dmg_expr = damage_dice or c.weapon[1]
            dmg_bonus = c.mod(ability)

        roll, detail = roll_d20(state.rng, advantage)
        atk_bonus = c.mod(ability) + c.prof_bonus
        total = roll + atk_bonus
        what = description or (c.weapon[0] if attack_type == "weapon" else "spell")

        if roll == 1 or (total < enemy.ac and roll != 20):
            return state.log(f"🗡️ {c.name} menyerang {enemy.name} ({what}): {detail} {fmt_mod(atk_bonus)} "
                             f"= **{total}** vs AC {enemy.ac} → MELESET")

        crit = roll == 20
        dmg, dmg_detail = roll_expression(dmg_expr, state.rng, crit=crit)
        dmg = max(1, dmg + dmg_bonus)
        enemy.hp = max(0, enemy.hp - dmg)
        msg = (f"🗡️ {c.name} menyerang {enemy.name} ({what}): {detail} {fmt_mod(atk_bonus)} = **{total}** "
               f"vs AC {enemy.ac} → {'CRITICAL HIT! 💥' if crit else 'KENA'} "
               f"Damage {dmg_detail}{' ' + fmt_mod(dmg_bonus) if dmg_bonus else ''} = **{dmg}**. "
               f"HP {enemy.name}: {enemy.hp}/{enemy.max_hp}")
        if enemy.hp == 0:
            msg += f". {enemy.name} KALAH!"
            state.enemies.pop(enemy.name, None)
            state.initiative_order = [x for x in state.initiative_order if x[0] != enemy.name]
            if enemy.xp:
                msg += f" (bernilai {enemy.xp} XP, berikan dengan award_xp)"
            if not state.enemies:
                msg += " Semua musuh sudah kalah, combat selesai."
                state.initiative_order = []
        return state.log(msg)

    @tool
    def damage_enemy(target: str, amount: int, reason: str) -> str:
        """Kurangi HP musuh di luar attack roll, mis. musuh gagal saving throw
        terhadap spell area. Untuk serangan biasa pakai player_attack.

        Args:
            target: nama musuh.
            amount: jumlah damage (positif). Lempar dulu dengan roll_dice.
            reason: penyebab damage.
        """
        enemy = state.find_enemy(target)
        if enemy is None:
            return f"ERROR: Musuh '{target}' tidak ditemukan."
        enemy.hp = max(0, enemy.hp - abs(amount))
        msg = f"💥 {enemy.name} terkena {abs(amount)} damage ({reason}). HP {enemy.hp}/{enemy.max_hp}"
        if enemy.hp == 0:
            msg += f". {enemy.name} KALAH!"
            state.enemies.pop(enemy.name, None)
            state.initiative_order = [x for x in state.initiative_order if x[0] != enemy.name]
            if enemy.xp:
                msg += f" (bernilai {enemy.xp} XP)"
        return state.log(msg)

    @tool
    def enemy_attack(enemy_name: str,
                     advantage: Literal["normal", "advantage", "disadvantage"] = "normal",
                     description: str = "") -> str:
        """Musuh menyerang karakter pemain memakai statistik musuh tersebut.
        Attack roll vs AC pemain dan damage dihitung otomatis; HP pemain langsung dikurangi.

        Args:
            enemy_name: nama musuh yang menyerang.
            advantage: 'advantage', 'disadvantage', atau 'normal'.
            description: deskripsi singkat serangannya.
        """
        if err := need_character():
            return err
        c = state.character
        enemy = state.find_enemy(enemy_name)
        if enemy is None:
            return f"ERROR: Musuh '{enemy_name}' tidak ditemukan."
        roll, detail = roll_d20(state.rng, advantage)
        total = roll + enemy.attack_bonus
        what = description or "serangan"
        if roll == 1 or (total < c.ac and roll != 20):
            return state.log(f"🛡️ {enemy.name} menyerang {c.name} ({what}): {detail} "
                             f"{fmt_mod(enemy.attack_bonus)} = **{total}** vs AC {c.ac} → MELESET")
        crit = roll == 20
        dmg, dmg_detail = roll_expression(enemy.damage_dice, state.rng, crit=crit)
        dmg = max(1, dmg)
        before = c.hp
        c.hp = max(0, c.hp - dmg)
        msg = (f"🩸 {enemy.name} menyerang {c.name} ({what}): {detail} {fmt_mod(enemy.attack_bonus)} "
               f"= **{total}** vs AC {c.ac} → {'CRITICAL HIT! 💥' if crit else 'KENA'} "
               f"Damage {dmg_detail} = **{dmg}**. HP {c.name}: {before} → {c.hp}/{c.max_hp}")
        if c.hp == 0 and "Unconscious" not in c.conditions:
            c.conditions.append("Unconscious")
            msg += ". TUMBANG! Mulai death saving throw."
        return state.log(msg)

    @tool
    def end_combat(outcome: str) -> str:
        """Akhiri combat (musuh kabur, menyerah, atau pemain melarikan diri).
        Menghapus semua musuh aktif dan urutan inisiatif.

        Args:
            outcome: ringkasan hasil combat.
        """
        state.enemies.clear()
        state.initiative_order = []
        return state.log(f"🏁 Combat selesai: {outcome}")

    # ── Referensi aturan (API eksternal) ─────────────────────────────────────
    @tool
    def lookup_srd(category: Literal["spells", "monsters", "conditions", "equipment",
                                     "magic-items", "classes", "races", "features",
                                     "rule-sections"],
                   name: str) -> str:
        """Cari aturan resmi D&D 5e SRD dari dnd5eapi.co (API eksternal).
        Gunakan saat pemain bertanya tentang spell, monster, kondisi, item, atau aturan,
        dan sebelum menerapkan spell/monster agar angkanya akurat.

        Args:
            category: kategori SRD.
            name: nama dalam bahasa Inggris, mis. 'Fireball', 'Goblin', 'Poisoned', 'Longsword'.
        """
        data = srd_find(category, name)
        if data is None:
            return (f"Tidak ditemukan '{name}' di kategori {category} SRD, atau API sedang "
                    f"tidak bisa diakses. Jika ini konten di luar SRD, jelaskan secara umum "
                    f"tanpa mengutip buku resmi.")
        return f"[Sumber: D&D 5e SRD via dnd5eapi.co]\n{format_srd(category, data)}"

    return [
        roll_dice, ability_check, create_character, get_game_state, update_hp,
        update_inventory, update_gold, set_condition, take_rest, award_xp,
        set_location, update_quest, add_enemy, roll_initiative, player_attack,
        damage_enemy, enemy_attack, end_combat, lookup_srd,
    ]

Overwriting game_tools.py


### Coba tools langsung, tanpa LLM

Mencoba tools dulu sebelum diberikan ke agent.

In [64]:
import importlib, random
import game_tools
importlib.reload(game_tools)   # pastikan memakai versi file terbaru
from game_tools import GameState, build_tools

state = GameState(rng=random.Random(42))
tools = {t.name: t for t in build_tools(state)}
print(f"{len(tools)} tools siap: {', '.join(tools)}\n")

print(tools["create_character"].invoke({
    "name": "Arya", "species": "Human", "char_class": "Fighter",
    "background": "pendekar desa pesisir"}))
print()
print(tools["roll_dice"].invoke({"notation": "2d6+3", "reason": "contoh lemparan"}))
print(tools["ability_check"].invoke({
    "ability": "DEX", "dc": 13, "skill_or_purpose": "Stealth", "advantage": "advantage"}))

19 tools siap: roll_dice, ability_check, create_character, get_game_state, update_hp, update_inventory, update_gold, set_condition, take_rest, award_xp, set_location, update_quest, add_enemy, roll_initiative, player_attack, damage_enemy, enemy_attack, end_combat, lookup_srd

Karakter berhasil dibuat.
Lokasi: Belum ditentukan
Karakter: Arya, Human Fighter level 1 (XP 0/300)
Background: pendekar desa pesisir
HP 12/12 | AC 18 (Chain Mail + Shield) | Proficiency +2 | Gold 15 gp | Hit dice tersisa 1d10
Ability: STR 15 (+2), DEX 13 (+1), CON 14 (+2), INT 8 (-1), WIS 12 (+1), CHA 10 (+0)
Senjata utama: Longsword (1d8 + STR)
Inventaris: Longsword x1, Shield x1, Chain Mail x1, Light Crossbow x1, Dungeoneer's Pack x1
Kondisi: normal
Musuh aktif: tidak ada (bukan mode combat)

🎲 contoh lemparan: 2d6+3 → 2d6[6, 1] + 3 = **10**
🎲 Check DEX (Stealth): d20 advantage [1, 9] → 9 +1 = **10** vs DC 13 → GAGAL ❌


In [65]:
# Simulasi combat singkat melawan makhluk orisinal
print(tools["add_enemy"].invoke({"name": "Genderuwo", "hp": 18, "ac": 12,
                                 "attack_bonus": 4, "damage_dice": "1d8+2"}))
print(tools["roll_initiative"].invoke({}))
print(tools["player_attack"].invoke({"target": "Genderuwo"}))
print(tools["enemy_attack"].invoke({"enemy_name": "Genderuwo"}))
print()
print(state.summary())

👹 Musuh muncul: Genderuwo (HP 18, AC 12, serangan +4 / 1d8+2) [custom]
⚔️ Inisiatif: Arya (9) → Genderuwo (8)
🗡️ Arya menyerang Genderuwo (Longsword): d20 [5] +4 = **9** vs AC 12 → MELESET
🛡️ Genderuwo menyerang Arya (serangan): d20 [4] +4 = **8** vs AC 18 → MELESET

Lokasi: Belum ditentukan
Karakter: Arya, Human Fighter level 1 (XP 0/300)
Background: pendekar desa pesisir
HP 12/12 | AC 18 (Chain Mail + Shield) | Proficiency +2 | Gold 15 gp | Hit dice tersisa 1d10
Ability: STR 15 (+2), DEX 13 (+1), CON 14 (+2), INT 8 (-1), WIS 12 (+1), CHA 10 (+0)
Senjata utama: Longsword (1d8 + STR)
Inventaris: Longsword x1, Shield x1, Chain Mail x1, Light Crossbow x1, Dungeoneer's Pack x1
Kondisi: normal
Musuh aktif: Genderuwo HP 18/18 AC 12
Urutan inisiatif:  Arya (9) →  Genderuwo (8)


In [66]:
# Integrasi API eksternal: ambil data monster dan spell dari D&D 5e SRD API
print(tools["lookup_srd"].invoke({"category": "monsters", "name": "Goblin"}))
print()
print(tools["lookup_srd"].invoke({"category": "spells", "name": "Fire Bolt"}))

[Sumber: D&D 5e SRD via dnd5eapi.co]
Goblin: Small humanoid, CR 0.25, XP 50
AC 15, HP 7 (2d6)
STR 8 DEX 14 CON 10 INT 10 WIS 8 CHA 8
Aksi: Scimitar (+4 to hit, 1d6+2); Shortbow (+4 to hit, 1d6+2)

[Sumber: D&D 5e SRD via dnd5eapi.co]
Fire Bolt (level 0, Evocation)
Casting time 1 action, range 120 feet, durasi Instantaneous, komponen VS
Damage: [{'damage_type': {'index': 'fire', 'name': 'Fire', 'url': '/api/2014/damage-types/fire'}, 'damage_at_character_level': {'1': '1d10', '5': '2d10', '11': '3d10', '17': '4d10'}}] 
You hurl a mote of fire at a creature or object within range. Make a ranged spell attack against the target. On a hit, the target takes 1d10 fire damage. A flammable object hit by this spell ignites if it isn't being worn or carried. This spell's damage increases by 1d10 when you reach 5th level (2d10), 11th level (3d10), and 17th level (4d10).


---
# Part 2: Agent Dungeon Master (`dm_agent.py`)

```python
agent = create_agent(
    model=ChatGoogleGenerativeAI(...),   # Gemini
    tools=[...],                          # dari game_tools.py
    system_prompt="...",                  # aturan dan persona DM
    checkpointer=InMemorySaver(),         # memory percakapan
)
```

- System prompt dirakit dari parameter kreatif oleh `build_system_prompt(settings)`.
  Ganti setting dunia di sidebar, prompt-nya ikut berubah.
- Prompt punya "aturan emas": DM dilarang mengarang angka dan wajib memanggil tools.
- Kalau pengaturan diubah di tengah permainan, agent dibangun ulang dengan prompt baru, tapi
  `checkpointer` dan `thread_id`-nya tetap. Jadi DM masih ingat ceritanya.
- `run_turn()` mengirim satu pesan, lalu memisahkan jawaban DM dari hasil tools. Hasil tools
  nanti ditampilkan di UI sebagai "🎲 Mekanik permainan".
- `message_text()` mengambil teks jawaban. Ini perlu karena Gemini kadang mengembalikan
  `content` dalam bentuk list, bukan string.

In [67]:
%%writefile dm_agent.py
"""
dm_agent.py
===========
Merakit "otak" Dungeon Master: system prompt yang dibentuk oleh parameter
kreatif (setting dunia, gaya narasi, tingkat kesulitan, bahasa, panjang
narasi) + model Gemini + tools dari game_tools.py + memory (checkpointer).

create_agent(model, tools, system_prompt, checkpointer)
"""

from __future__ import annotations

from dataclasses import dataclass

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver

from game_tools import GameState, build_tools

# ─────────────────────────────────────────────────────────────────────────────
# Parameter kreatif (pilihan ini muncul di sidebar Streamlit)
# ─────────────────────────────────────────────────────────────────────────────

WORLDS = {
    "Nusantara Mistis": (
        "Kepulauan fiktif bernama Swarnadwipa yang terinspirasi folklor Nusantara: hutan "
        "beringin keramat, candi tua yang ditelan akar, pasar malam di desa pesisir, "
        "kerajaan maritim, keris bertuah, dan makhluk seperti genderuwo, kuntilanak, "
        "leak, naga Antaboga, dan siluman harimau. Sihir disebut 'ilmu' dan dukun, "
        "resi, serta pendekar silat menggantikan penyihir dan kesatria. Perlakukan "
        "budaya dan kepercayaan lokal dengan hormat."
    ),
    "High Fantasy Klasik": (
        "Kerajaan fiktif Valdren: kastil batu, desa petani, hutan elf kuno, tambang "
        "dwarf di pegunungan, guild petualang, dan reruntuhan peradaban sihir yang "
        "runtuh. Nada petualangan klasik ala sword & sorcery."
    ),
    "Horor Gothic": (
        "Lembah Morrowgrave yang selalu berkabut: desa yang ketakutan, kastil bangsawan "
        "terkutuk, gereja tua, makam, dan makhluk malam. Horor lebih banyak dibangun "
        "lewat suasana dan ketegangan, bukan gore berlebihan."
    ),
    "Bajak Laut & Lautan": (
        "Kepulauan Saltreach: kota pelabuhan penuh penyelundup, kapal bajak laut, "
        "pulau harta karun, badai sihir, dan monster laut. Banyak intrik, negosiasi, "
        "dan aksi di atas geladak."
    ),
}

TONES = {
    "Epik & Dramatis": "Narasi sinematik dan penuh momen heroik; NPC berbicara dengan gaya yang berkesan.",
    "Santai & Kocak": "Narasi ringan dan jenaka, NPC eksentrik, banyak humor situasional, tapi bahaya tetap nyata.",
    "Gelap & Mencekam": "Narasi tegang dan atmosferik, sumber daya terbatas, pilihan moral yang sulit.",
}

DIFFICULTIES = {
    "Mudah": "Gunakan DC 8-12 untuk sebagian besar check. Musuh sedikit dan lemah (CR rendah). Beri banyak petunjuk.",
    "Normal": "Gunakan DC 10-15 untuk sebagian besar check. Musuh seimbang dengan level karakter.",
    "Sulit": "Gunakan DC 13-18 untuk sebagian besar check. Musuh taktis, sumber daya langka, kegagalan berkonsekuensi berat.",
}

LANGUAGES = {
    "Bahasa Indonesia": "Selalu menjawab dalam Bahasa Indonesia yang luwes dan hidup. Istilah mekanik D&D (HP, AC, DC, nama spell/class) boleh tetap bahasa Inggris.",
    "English": "Always respond in natural, vivid English.",
}

NARRATION_LENGTHS = {
    "Singkat": "Narasi 1 paragraf pendek per giliran.",
    "Sedang": "Narasi 1-3 paragraf per giliran.",
    "Panjang": "Narasi 3-5 paragraf yang kaya detail per giliran.",
}

MODELS = ["gemini-3.1-flash-lite"]


@dataclass(frozen=True)
class DMSettings:
    world: str = "Nusantara Mistis"
    tone: str = "Epik & Dramatis"
    difficulty: str = "Normal"
    language: str = "Bahasa Indonesia"
    narration: str = "Sedang"
    model: str = "gemini-3.1-flash-lite"
    temperature: float = 0.9


# ─────────────────────────────────────────────────────────────────────────────
# System prompt
# ─────────────────────────────────────────────────────────────────────────────

def build_system_prompt(s: DMSettings) -> str:
    return f"""
Kamu adalah **Sang Juru Kisah**, seorang Dungeon Master (DM) AI untuk permainan role-playing
bergaya Dungeons & Dragons 5e dengan SATU pemain. Tugasmu: menarasikan dunia, memerankan NPC,
dan menjalankan aturan secara adil sehingga pemain merasakan petualangan yang seru.

## Parameter sesi
- Setting dunia: {s.world}. {WORLDS[s.world]}
- Gaya narasi: {s.tone}. {TONES[s.tone]}
- Tingkat kesulitan: {s.difficulty}. {DIFFICULTIES[s.difficulty]}
- Bahasa: {LANGUAGES[s.language]}
- Panjang narasi: {NARRATION_LENGTHS[s.narration]}

## ATURAN EMAS: angka berasal dari tools, bukan dari kepalamu
1. JANGAN PERNAH mengarang hasil dadu, HP, AC, gold, atau isi inventaris. Selalu pakai tools:
   - ability_check untuk check/saving throw pemain; roll_dice untuk lemparan lain.
   - player_attack dan enemy_attack untuk serangan (otomatis mengurangi HP).
   - update_hp, update_inventory, update_gold, set_condition untuk perubahan lain.
   - award_xp setelah musuh kalah atau quest selesai.
2. Jika ragu tentang kondisi permainan, panggil get_game_state.
3. Laporkan hasil tool apa adanya di narasi (mis. "Kamu mendapat 14, cukup untuk memanjat").
   Jangan mengubah keberhasilan menjadi kegagalan atau sebaliknya.

## Alur sesi
- Awal sesi: panggil get_game_state. Jika karakter belum dibuat, sambut pemain dengan
  pembuka yang atmosferik sesuai setting, lalu tanyakan nama, spesies/ras, dan class
  (tawarkan pilihan class: Barbarian, Bard, Cleric, Druid, Fighter, Monk, Paladin, Ranger,
  Rogue, Sorcerer, Warlock, Wizard) dan latar belakang singkat. Untuk setting Nusantara,
  kamu boleh menyebut padanan lokal (mis. Fighter = pendekar, Cleric = resi), tetapi
  argumen char_class ke tool tetap nama bahasa Inggris. Setelah lengkap, panggil
  create_character, lalu mulai petualangan: set_location dan berikan quest pertama
  dengan update_quest.
- Setiap giliran: narasikan akibat aksi pemain, lalu akhiri dengan pertanyaan
  "Apa yang kamu lakukan?" (atau padanannya) ditambah 2-3 saran aksi singkat dalam
  bentuk daftar. Pemain tetap bebas melakukan hal lain.
- Minta ability_check hanya ketika hasilnya tidak pasti dan kegagalan menarik.
  Hal sepele berhasil otomatis.

## Combat
1. Tambahkan setiap musuh dengan add_enemy. Untuk monster yang ada di SRD (goblin, wolf,
   skeleton, zombie, bandit, orc, dll.), isi srd_monster agar statistik diambil dari API.
   Untuk makhluk orisinal setting (mis. genderuwo), tentukan hp/ac/attack_bonus/damage_dice
   yang seimbang dengan tingkat kesulitan dan level karakter.
2. Panggil roll_initiative sekali, lalu jalankan giliran sesuai urutan.
3. Giliran musuh: panggil enemy_attack untuk setiap musuh yang menyerang, dalam SATU respons,
   lalu kembalikan giliran ke pemain.
4. Setelah semua musuh kalah: award_xp (gunakan nilai XP dari hasil tool jika ada),
   beri loot yang wajar lewat update_inventory/update_gold.
5. Jika HP pemain 0: narasikan dengan dramatis dan jalankan death saving throw
   (roll_dice '1d20': ≥10 sukses, 3 sukses = stabil, 3 gagal = mati).

## Pengetahuan aturan
- Untuk pertanyaan aturan, spell, monster, item, atau kondisi, panggil lookup_srd lebih dulu
  dan jawab berdasarkan hasilnya.
- Hanya gunakan konten dari System Reference Document (SRD) 5e yang berlisensi CC-BY-4.0.
  Jangan memakai setting resmi (mis. Forgotten Realms), karakter resmi, atau monster
  non-SRD milik Wizards of the Coast; ciptakan versi orisinal sebagai gantinya.

## Gaya bermain
- Hormati agensi pemain: jangan memutuskan tindakan, ucapan, atau perasaan karakter pemain.
- Dunia bereaksi secara logis: tindakan ceroboh punya konsekuensi, ide kreatif dihargai.
- Pertanyaan out-of-character (OOC) tentang aturan atau karakter dijawab singkat dan jelas,
  lalu kembali ke cerita.
- Jika pemain meminta hal di luar permainan sama sekali, jawab singkat bahwa kamu adalah DM
  untuk petualangan ini dan tawarkan untuk kembali bermain.
- Konten kekerasan sebatas fantasi petualangan; hindari gore eksplisit dan konten seksual.
- Gunakan Markdown seperlunya: **tebal** untuk nama penting, daftar untuk pilihan aksi.
""".strip()


# ─────────────────────────────────────────────────────────────────────────────
# Agent
# ─────────────────────────────────────────────────────────────────────────────

def build_agent(api_key: str, settings: DMSettings, state: GameState, checkpointer: InMemorySaver):
    """Rakit agent DM. Checkpointer yang sama dipakai ulang agar memory percakapan
    tetap ada meskipun pengaturan (setting, model, temperature) diubah di tengah game."""
    model = ChatGoogleGenerativeAI(
        model=settings.model,
        temperature=settings.temperature,
        google_api_key=api_key,
        max_retries=2,
    )
    return create_agent(
        model=model,
        tools=build_tools(state),
        system_prompt=build_system_prompt(settings),
        checkpointer=checkpointer,
    )


def message_text(msg: BaseMessage) -> str:
    """Ambil teks dari pesan AI. Gemini kadang mengembalikan content berupa list of parts."""
    content = msg.content
    if isinstance(content, str):
        return content
    parts = []
    for part in content:
        if isinstance(part, str):
            parts.append(part)
        elif isinstance(part, dict) and part.get("type") == "text":
            parts.append(part.get("text", ""))
    return "".join(parts)


def run_turn(agent, user_text: str, thread_id: str) -> tuple[str, list[str]]:
    """Kirim satu pesan ke agent. Mengembalikan (jawaban_DM, daftar_hasil_tool)."""
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 40}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_text}]},
        config=config
    )
    messages = result["messages"]

    # Ambil hanya pesan dari giliran ini (setelah pesan user terakhir)
    start = 0
    for i in range(len(messages) - 1, -1, -1):
        if messages[i].type == "human":
            start = i + 1
            break
    turn = messages[start:]

    # get_game_state hanya dibaca oleh DM, tidak perlu ditampilkan ke pemain
    tool_results = [str(m.content) for m in turn
                    if isinstance(m, ToolMessage) and m.name != "get_game_state"]
    ai_texts = [message_text(m) for m in turn if isinstance(m, AIMessage)]
    answer = next((t for t in reversed(ai_texts) if t.strip()), "")
    if not answer:
        answer = "_(Sang Juru Kisah terdiam sejenak... coba ulangi aksimu.)_"
    return answer, tool_results

Overwriting dm_agent.py


---
# Part 3: Tampilan Streamlit (`app.py`)

Komponen yang dipakai sebagian besar sudah dibahas di materi Streamlit:

- `st.session_state` menyimpan `GameState`, `InMemorySaver`, `thread_id`, riwayat chat, dan agent,
  supaya tidak hilang setiap kali skrip dijalankan ulang.
- Sidebar berisi API key, parameter kreatif, pilihan model, tombol reset, dan character sheet.
- `st.chat_message` dan `st.chat_input` untuk bubble chat.
- `st.progress`, `st.metric`, dan `st.expander` untuk HP, XP, inventaris, quest log, dan log dadu.
- `st.stop()` kalau belum ada API key, `st.rerun()` saat petualangan di-reset.

Alur tiap kali skrip dijalankan:

```
Sidebar: API key + parameter kreatif
         ↓
Belum ada key? → st.stop()
         ↓
Bangun agent (hanya kalau key atau pengaturan berubah)
         ↓
Tampilkan riwayat chat. Kalau sesi baru, DM membuka cerita sendiri
         ↓
Input pemain (chat_input atau tombol aksi cepat)
         ↓
run_turn(): Gemini ↔ tools (dadu, HP, combat, SRD API)
         ↓
Tampilkan jawaban DM + "🎲 Mekanik permainan"
         ↓
Update character sheet di sidebar
```

Character sheet ditulis ke sebuah `st.container()` di sidebar pada bagian akhir skrip. Kalau
ditulis di awal, HP yang tampil masih HP sebelum giliran berjalan.

File `.streamlit/config.toml` berisi tema gelap dengan aksen emas.

In [68]:
!mkdir -p .streamlit

In [69]:
%%writefile .streamlit/config.toml
[theme]
base = "dark"
primaryColor = "#C9A227"
backgroundColor = "#14110F"
secondaryBackgroundColor = "#221C17"
textColor = "#EDE6D6"
font = "serif"

[server]
headless = true


Overwriting .streamlit/config.toml


In [70]:
%%writefile app.py
"""
app.py: Sang Juru Kisah, AI Dungeon Master
==========================================
UI Streamlit untuk chatbot D&D berbasis Gemini + LangGraph agent.

Jalankan:
    streamlit run app.py
"""

import os
import uuid

import streamlit as st
from langgraph.checkpoint.memory import InMemorySaver

from dm_agent import (DIFFICULTIES, LANGUAGES, MODELS, NARRATION_LENGTHS, TONES,
                      WORLDS, DMSettings, build_agent, run_turn)
from game_tools import ABILITIES, XP_THRESHOLDS, GameState, fmt_mod

# ── 1. Konfigurasi halaman ───────────────────────────────────────────────────
st.set_page_config(page_title="Sang Juru Kisah | AI Dungeon Master",
                   page_icon="🐉", layout="wide")

st.markdown(
    """
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Cinzel:wght@600;700&display=swap');
    h1, h2, h3 { font-family: 'Cinzel', serif !important; letter-spacing: .02em; }
    [data-testid="stChatMessage"] { border-radius: 12px; }
    .ability-grid { display:grid; grid-template-columns:repeat(3,1fr); gap:6px; margin:4px 0 8px; }
    .ability { border:1px solid rgba(201,162,39,.45); border-radius:8px; padding:4px 0; text-align:center; }
    .ability b { display:block; font-size:.72rem; opacity:.75; }
    .ability span { font-size:1.05rem; font-weight:700; }
    .ability small { display:block; font-size:.72rem; opacity:.8; }
    </style>
    """,
    unsafe_allow_html=True,
)

DM_AVATAR, PLAYER_AVATAR = "🧙", "🗡️"
KICKOFF = ("[SISTEM] Sesi baru dimulai dan pemain belum menulis apa pun. Buka petualangan "
           "sesuai parameter sesi, lalu mulai pembuatan karakter.")


# ── 2. Session state ─────────────────────────────────────────────────────────
# Streamlit menjalankan ulang skrip setiap interaksi, jadi semua yang harus
# bertahan (state game, memory agent, riwayat chat) disimpan di session_state.
def new_game() -> None:
    st.session_state.game = GameState()                 # state mekanik (HP, item, dll.)
    st.session_state.checkpointer = InMemorySaver()     # memory percakapan LangGraph
    st.session_state.thread_id = str(uuid.uuid4())
    st.session_state.messages = []                      # riwayat untuk ditampilkan
    st.session_state.agent = None
    st.session_state.agent_key = None
    st.session_state.started = False


if "game" not in st.session_state:
    new_game()


def default_api_key() -> str:
    """Ambil key dari env var atau .streamlit/secrets.toml jika ada (opsional)."""
    key = os.environ.get("GOOGLE_API_KEY", "")
    if key:
        return key
    try:
        return st.secrets.get("GOOGLE_API_KEY", "")
    except Exception:
        return ""


# ── 3. Sidebar: pengaturan & parameter kreatif ──────────────────────────────
with st.sidebar:
    st.header("⚙️ Pengaturan")
    api_key = st.text_input("Google AI API Key", type="password", value=default_api_key(),
                            help="Dapatkan gratis di https://aistudio.google.com")

    with st.expander("🎭 Parameter Kreatif", expanded=True):
        world = st.selectbox("Setting dunia", list(WORLDS))
        tone = st.selectbox("Gaya narasi", list(TONES))
        difficulty = st.select_slider("Tingkat kesulitan", list(DIFFICULTIES), value="Normal")
        language = st.radio("Bahasa", list(LANGUAGES), horizontal=True)
        narration = st.select_slider("Panjang narasi", list(NARRATION_LENGTHS), value="Sedang")

    with st.expander("🤖 Model AI"):
        model_name = st.selectbox("Model Gemini", MODELS)
        temperature = st.slider("Temperature (kreativitas)", 0.0, 1.5, 0.9, 0.05,
                                help="Rendah = konsisten & patuh aturan, tinggi = lebih imajinatif.")

    if st.button("🔄 Petualangan Baru", width="stretch",
                 help="Hapus karakter, memory, dan riwayat chat."):
        new_game()
        st.rerun()

    st.divider()
    sheet_box = st.container()   # character sheet diisi di akhir skrip (setelah DM menjawab)


# ── 4. Komponen tampilan ─────────────────────────────────────────────────────
def render_sheet(game: GameState) -> None:
    """Character sheet, musuh, quest, dan log dadu di sidebar."""
    with sheet_box:
        c = game.character
        if c is None:
            st.subheader("📜 Character Sheet")
            st.caption("Belum ada karakter. Ceritakan pada Sang Juru Kisah siapa dirimu!")
            return

        st.subheader(f"🛡️ {c.name}")
        st.caption(f"{c.species} {c.char_class} · Level {c.level}"
                   + (f" · {c.background}" if c.background else ""))
        st.progress(c.hp / c.max_hp if c.max_hp else 0, text=f"❤️ HP {c.hp}/{c.max_hp}")
        next_xp = XP_THRESHOLDS[c.level] if c.level < 20 else c.xp
        prev_xp = XP_THRESHOLDS[c.level - 1]
        xp_ratio = 1.0 if c.level >= 20 else (c.xp - prev_xp) / (next_xp - prev_xp)
        st.progress(min(1.0, max(0.0, xp_ratio)), text=f"⭐ XP {c.xp}/{next_xp}")

        col1, col2, col3 = st.columns(3)
        col1.metric("AC", c.ac)
        col2.metric("Gold", f"{c.gold}")
        col3.metric("Prof", fmt_mod(c.prof_bonus))

        cells = "".join(
            f"<div class='ability'><b>{a}</b><span>{c.abilities[a]}</span>"
            f"<small>{fmt_mod(c.mod(a))}</small></div>" for a in ABILITIES)
        st.markdown(f"<div class='ability-grid'>{cells}</div>", unsafe_allow_html=True)

        st.markdown(f"**Senjata:** {c.weapon[0]} ({c.weapon[1]})  \n"
                    f"**Armor:** {c.armor_name}  \n"
                    f"**Kondisi:** {', '.join(c.conditions) or 'normal'}  \n"
                    f"**Lokasi:** {game.location}")

        with st.expander(f"🎒 Inventaris ({sum(c.inventory.values())})"):
            for item, qty in c.inventory.items():
                st.markdown(f"- {item} × {qty}")

        if game.quests:
            with st.expander("📜 Quest Log", expanded=True):
                icons = {"aktif": "🟡", "selesai": "✅", "gagal": "❌"}
                for title, q in game.quests.items():
                    st.markdown(f"{icons.get(q['status'], '•')} **{title}**  \n"
                                f"<small>{q['note']}</small>", unsafe_allow_html=True)

        if game.enemies:
            st.markdown("#### ⚔️ Combat")
            for e in game.enemies.values():
                st.progress(e.hp / e.max_hp if e.max_hp else 0,
                            text=f"👹 {e.name} · HP {e.hp}/{e.max_hp} · AC {e.ac}")
            if game.initiative_order:
                st.caption("Giliran: " + " → ".join(n for n, _ in game.initiative_order))

    if game.roll_log:
        with sheet_box.expander("🎲 Log Dadu & Mekanik"):
            for line in reversed(game.roll_log[-12:]):
                st.markdown(line)


def render_message(msg: dict) -> None:
    avatar = DM_AVATAR if msg["role"] == "assistant" else PLAYER_AVATAR
    with st.chat_message(msg["role"], avatar=avatar):
        st.markdown(msg["content"])
        if msg.get("mechanics"):
            with st.expander(f"🎲 Mekanik permainan ({len(msg['mechanics'])})"):
                for line in msg["mechanics"]:
                    st.markdown(line)


def ask_dm(text: str) -> None:
    """Kirim teks ke agent DM, tampilkan jawabannya, lalu simpan ke riwayat."""
    with st.chat_message("assistant", avatar=DM_AVATAR):
        with st.spinner("Sang Juru Kisah sedang merangkai kisah..."):
            try:
                answer, mechanics = run_turn(st.session_state.agent, text,
                                             st.session_state.thread_id)
            except Exception as e:  # rate limit, key salah, koneksi, dll.
                answer, mechanics = f"⚠️ Terjadi error saat menghubungi Gemini:\n\n`{e}`", []
        st.markdown(answer)
        if mechanics:
            with st.expander(f"🎲 Mekanik permainan ({len(mechanics)})"):
                for line in mechanics:
                    st.markdown(line)
    st.session_state.messages.append(
        {"role": "assistant", "content": answer, "mechanics": mechanics})


# ── 5. Header ────────────────────────────────────────────────────────────────
st.title("🐉 Sang Juru Kisah")
st.caption(f"AI Dungeon Master · {world} · {tone} · {difficulty} · {model_name} "
           f"(temp {temperature})")

if not api_key:
    st.info("Masukkan Google AI API Key di sidebar untuk memulai petualangan.", icon="🗝️")
    render_sheet(st.session_state.game)
    st.stop()

# ── 6. Agent: dibangun ulang hanya jika key/pengaturan berubah ───────────────
settings = DMSettings(world=world, tone=tone, difficulty=difficulty, language=language,
                      narration=narration, model=model_name, temperature=temperature)
agent_key = (api_key, settings)
if st.session_state.agent_key != agent_key:
    try:
        st.session_state.agent = build_agent(api_key, settings, st.session_state.game,
                                             st.session_state.checkpointer)
    except Exception as e:
        st.error(f"Gagal menyiapkan model: {e}")
        st.stop()
    if st.session_state.agent_key is not None:
        st.toast("Pengaturan DM diperbarui. Memory petualangan tetap tersimpan.", icon="✨")
    st.session_state.agent_key = agent_key

# ── 7. Riwayat & pembuka otomatis ────────────────────────────────────────────
# Semua bubble chat ditaruh di container ini supaya tombol aksi cepat selalu
# berada di bawah percakapan.
chat_area = st.container()
with chat_area:
    for msg in st.session_state.messages:
        render_message(msg)

    if not st.session_state.started:
        st.session_state.started = True
        ask_dm(KICKOFF)

# ── 8. Aksi cepat & input pemain ─────────────────────────────────────────────
QUICK_ACTIONS = {
    "🔍 Amati sekitar": "Aku mengamati sekelilingku dengan saksama.",
    "📋 Status karakter": "(OOC) Tolong ringkas kondisi karakterku dan quest yang sedang aktif.",
    "🏕️ Istirahat sejenak": "Aku mencari tempat aman untuk short rest.",
    "❓ Tanya aturan": "(OOC) Jelaskan secara singkat aksi apa saja yang bisa kulakukan sekarang.",
}
quick_prompt = None
cols = st.columns(len(QUICK_ACTIONS))
for col, (label, text) in zip(cols, QUICK_ACTIONS.items()):
    if col.button(label, width="stretch"):
        quick_prompt = text

prompt = st.chat_input("Apa yang kamu lakukan?") or quick_prompt
if prompt:
    user_msg = {"role": "user", "content": prompt}
    st.session_state.messages.append(user_msg)
    with chat_area:
        render_message(user_msg)
        ask_dm(prompt)

# ── 9. Character sheet (di akhir agar mencerminkan hasil giliran ini) ───────
render_sheet(st.session_state.game)


Overwriting app.py


---
# Jalankan Chatbot

Jalankan cell di bawah, lalu buka URL ngrok yang muncul. Kalau keluar halaman peringatan dari
ngrok, klik **Visit Site**.

In [71]:
proc = run_streamlit("app.py")

Streamlit berjalan: NgrokTunnel: "https://underuse-maximize-rebirth.ngrok-free.dev" -> "http://localhost:8501"


### Cara mencoba

1. Buka URL di atas. API key seharusnya sudah terisi di sidebar. Kalau belum, ketik manual.
2. Pilih setting dunia, gaya narasi, dan tingkat kesulitan.
3. DM akan membuka cerita dan menanyakan karaktermu. Contoh jawaban: *"Namaku Laras, manusia
   Rogue, mantan pencopet dari pasar Tuban."* Character sheet akan muncul di sidebar.
4. Coba beberapa jenis aksi:
   - *"Aku menyelinap melewati penjaga candi."* untuk memicu ability check
   - *"Aku menyerang goblin itu dengan pedangku!"* untuk memulai combat
   - *"(OOC) Apa efek kondisi Poisoned?"* untuk mengambil data dari SRD API
5. Buka expander 🎲 Mekanik permainan di bawah jawaban DM untuk melihat hasil tools.
6. Ganti setting di tengah permainan. Gaya DM berubah, tapi ceritanya tetap diingat.

Untuk screenshot deliverable, ambil momen pembukaan dan pembuatan karakter, ability check, dan
combat (HP bar musuh terlihat di sidebar). Kalau mau, tambahkan satu screenshot dengan setting
dunia yang berbeda.

### Kalau ada masalah

- **`ERR_NGROK_108` atau batas tunnel**: akun ngrok gratis hanya boleh punya satu tunnel. Jalankan cell "Hentikan App", lalu ulangi.
- **Error 429 atau quota**: batas gratis Gemini habis sementara. Tunggu sebentar atau ganti model ke `gemini-2.5-flash-lite`.
- **API key not valid**: cek lagi key di sidebar atau di secret `GEMINI`.
- **Halaman kosong**: tunggu beberapa detik lalu refresh.
- **Monster SRD tidak ditemukan**: pakai nama bahasa Inggris, misalnya "goblin" atau "wolf". Kalau API gagal, DM akan membuat statistik sendiri.

---

## Menghentikan App

Jalankan cell di bawah untuk menghentikan Streamlit dan menutup tunnel ngrok.

In [72]:
# Hentikan Streamlit
try:
    proc.terminate()
    print("Streamlit dihentikan.")
except:
    print("Tidak ada proses yang berjalan.")

# Tutup semua tunnel ngrok
ngrok.kill()
print("Tunnel ngrok ditutup.")

Streamlit dihentikan.
Tunnel ngrok ditutup.


## Ringkasan

| Bagian | Yang dipakai |
|---|---|
| LLM | Gemini lewat `langchain-google-genai` |
| Agent | `create_agent` dari LangChain (LangGraph) |
| Memory | `InMemorySaver` + `thread_id` |
| Tools | 19 fungsi di `game_tools.py` |
| API eksternal | D&D 5e SRD API (dnd5eapi.co) |
| UI | Streamlit |
| Akses publik | Google Colab + ngrok |

## Lisensi dan atribusi

This work includes material taken from the System Reference Document 5.1 ("SRD 5.1") by Wizards of
the Coast LLC and available at https://dnd.wizards.com/resources/systems-reference-document. The SRD
5.1 is licensed under the Creative Commons Attribution 4.0 International License available at
https://creativecommons.org/licenses/by/4.0/legalcode.

Data aturan diambil dari [D&D 5e SRD API](https://www.dnd5eapi.co). Proyek ini tidak berafiliasi
dengan Wizards of the Coast.